
# 5.1 章节介绍：非量化 FA 开发

欢迎来到课程的**核心实战章节**！

第 4 章我们把官方 FA 算子跑起来、调测通过了。本章开始，我们自己写一个——**以渐进式的方式从最简实现一步步逼近工业级 Kernel**，每一阶段都是一次可运行、可验证的代码迭代。

> 本章聚焦**非量化** FA：输入 Q、K、V 均为 FP16/BF16，Softmax 在 FP32 上累加，全过程不涉及量化/反量化（量化版本见第 6 章）。

## 七阶段开发路线

每个阶段在前一阶段的基础上只引入**一个新概念**，规格固定为：输入只有 QKV 和 scale，输出 atten，VF 为 softmax，BSND 排布，非量化，D=128：

| 阶段 | 小节 | 新增概念 | buffer | 流水 | 分核 | mask |
|--|--|--|--|--|--|--|
| 一 | [5.3 基础极简 FA](05.03_basic_fa.ipynb) | 最小可运行 | 单 buffer | 串行 | 单核 | 无 |
| 二 | [5.4 BN1 简单分核](05.04_bn1_split.ipynb) | 多核并行 | 单 buffer | 串行 | BN1 | 无 |
| 三 | [5.5 buffer 复用](05.05_pingpong.ipynb) | 乒乓缓冲 | Q 常驻 + KV pingpong | 串行 | BN1 | 无 |
| 四 | [5.6 preload 流水](05.06_preload.ipynb) | 流水编排 | Q 常驻 + KV pingpong | preload | BN1 | 无 |
| 五 | [5.7 mask](05.07_mask.ipynb) | 掩码支持 | 同上 | preload | BN1 | sparsemode3 |
| 六 | [5.8 QS/KVS 带尾块](05.08_tail_block.ipynb) | 变长处理 | 同上 | preload | BN1 | sparsemode3 |
| 七 | [5.9 负载均衡](05.09_load_balance.ipynb) | 任务均衡 | 同上 | preload | BN1S1 | sparsemode3 |

## 本章学习目标

1. **写出最小 FA Kernel**：单核、串行、D=128 等长的四拍循环（C1→V1→C2→V2）。
2. **掌握分核**：BN1 切法的核间切块原则（无数据依赖）；理解 FlashDecode 为何改切 KV。
3. **掌握 buffer 管理**：Q 常驻 + KV 乒乓，让搬运与计算重叠。
4. **掌握流水编排**：串行 → N Buffer → Preload 的演进逻辑；理解「用两个 Cube 阶段掩盖一个较长 Vector 阶段」的原理。
5. **支持完整特性**：mask（sparsemode3）、QS/KVS 尾块、BN1S1 负载均衡。

## 前置知识

- [第2章 FlashAttention 原理](../02_fa_principle/README.md)（Online Softmax 递推）
- [第3章 FA 算子代码结构](../03_fa_code_architecture/README.md)（Tiling/Kernel 分层）
- [第4章 FA 算子运行](../04_fa_operator_running/README.md)（调测四步流程）

## 章节内容导航

| Notebook | 主题 |
|--|--|
| [5.1 章节介绍](05.01_chapter_intro.ipynb)（本页） | 导学 |
| [5.2 非量化 FA 介绍](05.02_flash_attention_introduction.ipynb) | 算子规格、计算流程、实现思路 |
| [5.3 基础极简 FA](05.03_basic_fa.ipynb) | 阶段一 |
| [5.4 BN1 简单分核](05.04_bn1_split.ipynb) | 阶段二 |
| [5.5 buffer 复用（pingpong）](05.05_pingpong.ipynb) | 阶段三 |
| [5.6 preload 流水](05.06_preload.ipynb) | 阶段四 |
| [5.7 mask](05.07_mask.ipynb) | 阶段五 |
| [5.8 QS 和 KVS 带尾块](05.08_tail_block.ipynb) | 阶段六 |
| [5.9 负载均衡](05.09_load_balance.ipynb) | 阶段七 |
| [5.10 章节测验](05.10_chapter_test.ipynb) | 综合练习 |

## 学习建议

- 每阶段先跑通再优化：正确性用第 4 章的 golden 对拍流程验证。
- 性能观察重点：**断流**。每个阶段改进后，问自己「Cube/Vector/搬运还有多少空等？」
- 建议把 5.6 节的流水模拟器思路自己实现一遍（Python threading 模拟 AIC/AIV），直观感受编排差异。
